# Analisis Kegagalan DDPG — PURE3 (StationVectorHead)

Notebook ini membuktikan seluruh lapis gejala kegagalan DDPG (dibanding PPO) LANGSUNG dari data pelatihan/evaluasi PURE3 (3 aliran: wait/gini/acceptance), arsitektur `StationVectorHead` (SVH), 2 horizon (30 & 90 hari).

Berkas sumber (semua ter-*commit* & ter-verifikasi):
- `master_pure_{ppo,ddpg}_pure3_svh_dgr[_90d]_acc1_training_results.json`
- `master_pure_{ppo,ddpg}_pure3_svh_specialist{0,1,2}_{wait,gini,accept}[_90d]_acc1_training_results.json`
- `uji_master_pure_{ppo,ddpg}_pure3_svh[_90d]_dgr_acc1_metrik_{30d,90d}.json` (evaluasi penuh 10-seed)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

OUT = "outputs"

def muat(f):
    return json.load(open(f"{OUT}/{f}", encoding="utf-8"))

def _gini(a):
    a = np.clip(np.asarray(a, dtype=float), 0, None)
    if a.sum() == 0:
        return 0.0
    a = np.sort(a)
    n = a.shape[0]
    idx = np.arange(1, n + 1)
    return float(np.sum((2 * idx - n - 1) * a) / (n * np.sum(a)))

NAMA_ALIRAN = ["wait", "gini", "accept"]

# --- DGR (30 hari & 90 hari) ---
PPO_DGR_30 = muat("master_pure_ppo_pure3_svh_dgr_acc1_training_results.json")
DDPG_DGR_30 = muat("master_pure_ddpg_pure3_svh_dgr_acc1_training_results.json")
PPO_DGR_90 = muat("master_pure_ppo_pure3_svh_dgr_90d_acc1_training_results.json")
DDPG_DGR_90 = muat("master_pure_ddpg_pure3_svh_dgr_90d_acc1_training_results.json")

# --- Spesialis (30 hari) ---
PPO_SPEC_30 = {
    "wait": muat("master_pure_ppo_pure3_svh_specialist0_wait_acc1_training_results.json"),
    "gini": muat("master_pure_ppo_pure3_svh_specialist1_gini_acc1_training_results.json"),
    "accept": muat("master_pure_ppo_pure3_svh_specialist2_accept_acc1_training_results.json"),
}
DDPG_SPEC_30 = {
    "wait": muat("master_pure_ddpg_pure3_svh_specialist0_wait_acc1_training_results.json"),
    "gini": muat("master_pure_ddpg_pure3_svh_specialist1_gini_acc1_training_results.json"),
    "accept": muat("master_pure_ddpg_pure3_svh_specialist2_accept_acc1_training_results.json"),
}

# --- Spesialis (90 hari) ---
PPO_SPEC_90 = {
    "wait": muat("master_pure_ppo_pure3_svh_specialist0_wait_90d_acc1_training_results.json"),
    "gini": muat("master_pure_ppo_pure3_svh_specialist1_gini_90d_acc1_training_results.json"),
    "accept": muat("master_pure_ppo_pure3_svh_specialist2_accept_90d_acc1_training_results.json"),
}
DDPG_SPEC_90 = {
    "wait": muat("master_pure_ddpg_pure3_svh_specialist0_wait_90d_acc1_training_results.json"),
    "gini": muat("master_pure_ddpg_pure3_svh_specialist1_gini_90d_acc1_training_results.json"),
    "accept": muat("master_pure_ddpg_pure3_svh_specialist2_accept_90d_acc1_training_results.json"),
}

# --- Evaluasi penuh (metrik 10-seed) ---
PPO_METRIK_30 = muat("uji_master_pure_ppo_pure3_svh_dgr_acc1_metrik_30d.json")
DDPG_METRIK_30 = muat("uji_master_pure_ddpg_pure3_svh_dgr_acc1_metrik_30d.json")
PPO_METRIK_90 = muat("uji_master_pure_ppo_pure3_svh_dgr_90d_acc1_metrik_90d.json")
DDPG_METRIK_90 = muat("uji_master_pure_ddpg_pure3_svh_dgr_90d_acc1_metrik_90d.json")

print("Semua berkas berhasil dimuat.")

## Lapis 0 — Metrik akhir (gini_UTIL, wait, acc), kondisi `signed|dinamis`

In [ ]:
def ambil_kondisi(d, kondisi="signed|dinamis"):
    a = d["agregat"]
    k = [kk for kk in a if kondisi in kk][0]
    return a[k]

def gini_util(d, kondisi="signed|dinamis"):
    ps = d["per_seed"]
    k = [kk for kk in ps if kondisi in kk][0]
    runs = ps[k]
    gu = np.array([_gini([v["util_mean"] for v in r["_stasiun"].values()]) for r in runs])
    if "ckpt_per_seed" in d and d.get("n_checkpoint"):
        grup = {c: [] for c in range(d["n_checkpoint"])}
        for sd_str, c in d["ckpt_per_seed"].items():
            grup[c].append(gu[int(sd_str)])
        gu = np.array([np.mean(v) for v in grup.values()])
    return float(gu.mean()), float(gu.std())

for label, dp, dd in [("30 hari", PPO_METRIK_30, DDPG_METRIK_30), ("90 hari", PPO_METRIK_90, DDPG_METRIK_90)]:
    rp, rd = ambil_kondisi(dp), ambil_kondisi(dd)
    gup, gup_sd = gini_util(dp)
    gud, gud_sd = gini_util(dd)
    print(f"=== {label} ===")
    print(f"                 PPO                  DDPG")
    print(f"gini_UTIL        {gup:.4f} ± {gup_sd:.4f}      {gud:.4f} ± {gud_sd:.4f}")
    print(f"wait (menit)     {rp['wait']:.1f} ± {rp['wait_sd']:.1f}       {rd['wait']:.1f} ± {rd['wait_sd']:.1f}")
    print(f"acc              {rp['acc']:.4f} ± {rp['acc_sd']:.4f}      {rd['acc']:.4f} ± {rd['acc_sd']:.4f}")
    print()

## Lapis 1 — Reward gini nyaris nol: `ret_mean` aliran gini (indeks 1), 10 iterasi terakhir

In [ ]:
def ret_mean_gini(runs):
    out = []
    for run in runs:
        rm = np.array([it["ret_mean"] for it in run["history"]])
        out.append((run["seed"], float(rm[-10:, 1].mean())))
    return out

print("=== PPO DGR: ret_mean aliran gini (10 iter akhir) ===")
print("  30d:", ret_mean_gini(PPO_DGR_30))
print("  90d:", ret_mean_gini(PPO_DGR_90))
print()
print("=== Spesialis gini SENDIRI (r_star, hasil AKHIR pelatihan) ===")
for label, spec in [("PPO 30d", PPO_SPEC_30["gini"]), ("PPO 90d", PPO_SPEC_90["gini"]),
                     ("DDPG 30d", DDPG_SPEC_30["gini"]), ("DDPG 90d", DDPG_SPEC_90["gini"])]:
    for run in spec:
        rstar = run.get("r_star")
        print(f"  {label} seed{run['seed']}: r_star={rstar}")

## Lapis 2 — Bobot `beta` DGR: apakah gini mendominasi TANPA substansi?

In [ ]:
def beta_akhir(runs, k=10):
    out = []
    for run in runs:
        b = np.array([it["beta"] for it in run["history"]])
        out.append((run["seed"], b[-k:].mean(axis=0)))
    return out

for label, runs in [("PPO DGR 30d", PPO_DGR_30), ("PPO DGR 90d", PPO_DGR_90),
                     ("DDPG DGR 30d", DDPG_DGR_30), ("DDPG DGR 90d", DDPG_DGR_90)]:
    print(f"=== {label} ===")
    for seed, b in beta_akhir(runs):
        print(f"  seed{seed}: wait={b[0]:.3f}  gini={b[1]:.3f}  accept={b[2]:.3f}")
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
konfig = [("PPO DGR 30d", PPO_DGR_30, axes[0,0]), ("DDPG DGR 30d", DDPG_DGR_30, axes[0,1]),
          ("PPO DGR 90d", PPO_DGR_90, axes[1,0]), ("DDPG DGR 90d", DDPG_DGR_90, axes[1,1])]
for judul, runs, ax in konfig:
    for run in runs:
        b = np.array([it["beta"] for it in run["history"]])
        ax.plot(b[:, 1], alpha=0.7, label=f"seed{run['seed']}")
    ax.set_title(f"{judul} -- beta_gini")
    ax.set_xlabel("iterasi"); ax.legend(fontsize=7)
axes[0,0].set_ylabel("beta_gini"); axes[1,0].set_ylabel("beta_gini")
plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_pure3_beta_gini.png", dpi=110)
plt.show()

## Lapis 3 — Eksplorasi: entropi PPO (konstan) vs noise_std DDPG (meluruh)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for run in PPO_DGR_90:
    ent = [it["entropy"] for it in run["history"]]
    axes[0].plot(ent, label=f"seed{run['seed']}")
axes[0].set_title("PPO DGR 90d -- entropi"); axes[0].set_xlabel("iterasi"); axes[0].legend()
for run in DDPG_DGR_90:
    ns = [it["noise_std"] for it in run["history"]]
    axes[1].plot(ns, label=f"seed{run['seed']}")
axes[1].set_title("DDPG DGR 90d -- noise_std"); axes[1].set_xlabel("iterasi"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/_notebook_pure3_eksplorasi.png", dpi=110)
plt.show()

## Lapis 4 — Grad_norm/critic_grad: lonjakan tak stabil

In [ ]:
print("=== grad_norm (PPO) / critic_grad (DDPG) MAKSIMUM, 90 hari ===")
for run in PPO_DGR_90:
    gn = [it["grad_norm"] for it in run["history"]]
    print(f"  PPO  seed{run['seed']}: maks={max(gn):.2f} median={np.median(gn):.2f}")
for run in DDPG_DGR_90:
    cg = [it["critic_grad"] for it in run["history"]]
    ag = [it["actor_grad"] for it in run["history"]]
    print(f"  DDPG seed{run['seed']}: critic_grad maks={max(cg):.2f} median={np.median(cg):.2f} | actor_grad maks={max(ag):.4g}")

## Lapis 5 — Konvergensi DEGENERATE di 90 hari: ketiga spesialis DDPG menyatu ke nilai sama?

In [ ]:
def ringkas_spesialis(spec_dict, label):
    print(f"=== {label} ===")
    for nama, runs in spec_dict.items():
        for run in runs:
            h = run["history"][-1]
            print(f"  spesialis={nama:8s} seed{run['seed']}: gini_util={h['gini_util']:.4f}")

ringkas_spesialis(DDPG_SPEC_90, "DDPG spesialis, 90 hari (cek: apakah wait/gini/accept konvergen ke angka SAMA?)")
print()
ringkas_spesialis(PPO_SPEC_90, "PPO spesialis, 90 hari (pembanding)")

## Lapis 6 — DGR vs spesialis-spesialisnya sendiri: kompromi wajar atau lebih buruk dari SEMUA?

In [ ]:
def wait_eval(d, kondisi="signed|dinamis"):
    return ambil_kondisi(d, kondisi)["wait"]

print("DGR (evaluasi penuh, wait menit):")
print(f"  PPO  30d={wait_eval(PPO_METRIK_30):.1f}   90d={wait_eval(PPO_METRIK_90):.1f}")
print(f"  DDPG 30d={wait_eval(DDPG_METRIK_30):.1f}   90d={wait_eval(DDPG_METRIK_90):.1f}")
print()
print("(Bandingkan dgn gini_util AKHIR pelatihan spesialis di Lapis 5 --")
print(" kalau DGR wait/gini_util LEBIH BURUK drpd SEMUA spesialisnya sendiri,")
print(" itu bukan kompromi wajar, itu kegagalan penggabungan.)")

## Ringkasan

Jalankan notebook ini sekali secara penuh -- seluruh angka & grafik (`_notebook_pure3_*.png`) akan
tersimpan otomatis di `outputs/`, siap dipakai langsung sbg bukti Lampiran/materi sidang.

**Kerangka lapis bukti:**
0. Metrik akhir -- gejala permukaan.
1. `ret_mean`/`r_star` gini ≈0 -- reward gini lemah (BERLAKU utk PPO & DDPG, KEDUA horizon).
2. Bobot `beta` -- apakah gini "mencuri" gradien tanpa substansi.
3. Eksplorasi -- entropi PPO konstan vs noise DDPG meluruh.
4. grad_norm/critic_grad -- lonjakan tak stabil.
5. Konvergensi degenerate -- KHUSUS 90 hari, cek apakah DDPG kolaps ke 1 solusi tak-terbedakan.
6. DGR vs spesialis -- kompromi wajar (PPO) atau lebih buruk dari semuanya (DDPG)?